# Comparación de modelos preentrenados de Hugging Face para clasificar tickets bancarios en español

**Proyecto:** asistente inteligente para clasificación y priorización de tickets de servicio al cliente.  
**Rama:** procesamiento de lenguaje natural (PLN).  
**Entorno previsto:** Google Colab con GPU.  
**Objetivo:** comparar tres codificadores de texto multilingües mediante el mismo conjunto de datos, el mismo clasificador final y las mismas métricas.

## Enfoque experimental

Se utiliza la versión en español de **Banking77 traducido** (`DeepPavlov/banking77-translated`, configuración `es`). Los modelos convierten cada mensaje en un vector semántico (*embedding*); después, un clasificador `LogisticRegression` aprende a asignar una de 77 intenciones bancarias. Los codificadores permanecen congelados: este ajuste mínimo permite comparar las representaciones sin tener que afinar tres redes neuronales completas.

**Importante:** el conjunto de datos es una traducción de preguntas de banca digital, no un histórico real de un banco mexicano. Los resultados son académicos y sirven para seleccionar candidatos; antes de producción deben validarse con tickets reales anonimizados, revisar dialectos locales y hacer análisis de privacidad y sesgos.

### Antes de ejecutar

1. En Colab, abre **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** (o la GPU disponible).
2. Ejecuta las celdas de arriba hacia abajo. La primera ejecución descarga dependencias, datos y pesos desde Hugging Face.
3. Los recursos de este proyecto son públicos; la autenticación no es obligatoria. La celda de autenticación permite usar `HF_TOKEN` guardado como secreto de Colab si se requiere acceso autenticado o se alcanza un límite.
4. Los resultados reales (`CSV` y gráfica) se generan al ejecutar el experimento; no se incluyen cifras inventadas.

In [ ]:
%pip -q install -U "datasets>=3.0.0" "sentence-transformers>=3.0.0" "scikit-learn>=1.4.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "huggingface_hub>=0.26.0" "safetensors"

In [ ]:
import os, json, time, random, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

SEED = 42
MAX_TRAIN_PER_CLASS = 80   # 77 clases x hasta 80 ejemplos = 6,160 ejemplos
MAX_TEST_PER_CLASS = 25    # 77 clases x hasta 25 ejemplos = 1,925 ejemplos
BATCH_SIZE = 64
LATENCY_SAMPLE_SIZE = 300       # tickets evaluados individualmente por modelo; p99 sigue siendo exploratorio
LATENCY_WARMUP_SAMPLES = 10     # llamadas de calentamiento, excluidas de los tiempos
MIN_F1_MACRO = 0.75             # umbral de aceptación propuesto; validar con el negocio
MAX_SINGLE_TICKET_P95_MS = 250  # SLA inicial propuesto para p95; confirmar con Operación
DATASET_ID = "DeepPavlov/banking77-translated"
DATASET_CONFIG = "es"
TEXT_FIELD = "text_gemma" # traducción al español; se mantienen las etiquetas del Banking77 original
OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True, parents=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    device = "cuda"
else:
    device = "cpu"

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("ADVERTENCIA: no se detectó GPU. El experimento funcionará en CPU pero será más lento.")

## 1. Autenticación opcional en Hugging Face Hub

Los modelos y datos seleccionados son públicos, por lo que se pueden descargar sin token. Si deseas dejar registrada una sesión autenticada, guarda un token de solo lectura llamado `HF_TOKEN` en **Colab → Secrets** y ejecuta la siguiente celda. No escribas el token directamente en el notebook ni lo subas a GitHub.

In [ ]:
HF_AUTHENTICATED = False
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    HF_AUTHENTICATED = True
    print("Sesión de Hugging Face autenticada mediante secreto.")
else:
    print("Sin token: se utilizarán únicamente recursos públicos. La autenticación es opcional para este prototipo.")

## 2. Carga y descripción del dataset

Campos relevantes de `banking77-translated`:

- `text`: consulta original en inglés.
- `text_gemma`: versión traducida al español por Gemma; será la entrada principal.
- `text_qwen`: otra versión traducida al español disponible para análisis de sensibilidad.
- `label`: identificador entero de una de las 77 intenciones bancarias.

Se respetan las particiones de entrenamiento y prueba publicadas por el dataset. Para controlar la duración de la sesión se toma una muestra balanceada por clase con semilla fija. No se mezclan filas entre particiones.

In [ ]:
raw_ds = load_dataset(DATASET_ID, DATASET_CONFIG)
print(raw_ds)
print("Columnas:", raw_ds["train"].column_names)
print("Features:", raw_ds["train"].features)
assert TEXT_FIELD in raw_ds["train"].column_names, f"No existe {TEXT_FIELD}; revisar las columnas disponibles."
assert "label" in raw_ds["train"].column_names, "No se encontró la columna label."

# Obtener nombres canónicos de clases desde el dataset original si los campos traducidos no los exponen.
label_names = None
try:
    feature = raw_ds["train"].features["label"]
    label_names = list(feature.names) if getattr(feature, "names", None) else None
except Exception:
    label_names = None
if not label_names:
    try:
        original_ds = load_dataset("PolyAI/banking77")
        feature = original_ds["train"].features["label"]
        label_names = list(feature.names) if getattr(feature, "names", None) else None
    except Exception as exc:
        print("No fue posible leer nombres canónicos de PolyAI; se mostrarán IDs numéricos.", exc)

# La versión traducida puede exponer `label` como entero sin `ClassLabel.names`.
# Si la carga de PolyAI falla, conservar los nombres oficiales según el ID 0-76.
BANKING77_INTENTS = ['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_physical_card', 'getting_spare_card', 'getting_virtual_card', 'lost_or_stolen_card', 'lost_or_stolen_phone', 'order_physical_card', 'passcode_forgotten', 'pending_card_payment', 'pending_cash_withdrawal', 'pending_top_up', 'pending_transfer', 'pin_blocked', 'receiving_money', 'Refund_not_showing_up', 'request_refund', 'reverted_card_payment?', 'supported_cards_and_currencies', 'terminate_account', 'top_up_by_bank_transfer_charge', 'top_up_by_card_charge', 'top_up_by_cash_or_cheque', 'top_up_failed', 'top_up_limits', 'top_up_reverted', 'topping_up_by_card', 'transaction_charged_twice', 'transfer_fee_charged', 'transfer_into_account', 'transfer_not_received_by_recipient', 'transfer_timing', 'unable_to_verify_identity', 'verify_my_identity', 'verify_source_of_funds', 'verify_top_up', 'virtual_card_not_working', 'visa_or_mastercard', 'why_verify_identity', 'wrong_amount_of_cash_received', 'wrong_exchange_rate_for_cash_withdrawal']
if not label_names or len(label_names) != 77:
    label_names = BANKING77_INTENTS
    print("Se utilizará el mapeo canónico de Banking77 (77 intenciones).")
else:
    print(f"Nombres de intención recuperados: {len(label_names)} clases.")

In [ ]:
def dataset_to_frame(split, text_field=TEXT_FIELD):
    df = pd.DataFrame(split[:])
    df = df[[text_field, "label"]].rename(columns={text_field: "text"})
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.len() > 0].copy()
    df["label"] = df["label"].astype(int)
    return df

def balanced_cap(df, max_per_class, seed=SEED):
    chunks = []
    for label, group in df.groupby("label", sort=True):
        n = min(max_per_class, len(group))
        chunks.append(group.sample(n=n, random_state=seed))
    out = pd.concat(chunks, ignore_index=True)
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)

train_all = dataset_to_frame(raw_ds["train"])
test_all = dataset_to_frame(raw_ds["test"])
train_df = balanced_cap(train_all, MAX_TRAIN_PER_CLASS)
test_df = balanced_cap(test_all, MAX_TEST_PER_CLASS)

train_labels = set(train_df["label"])
test_labels = set(test_df["label"])
if train_labels != test_labels:
    missing_train = sorted(test_labels - train_labels)
    missing_test = sorted(train_labels - test_labels)
    raise ValueError(f"Particiones no contienen las mismas clases. Falta train={missing_train}, test={missing_test}. Ajusta el máximo por clase.")

if label_names:
    train_df["intent_name"] = train_df["label"].map(lambda x: label_names[int(x)] if 0 <= int(x) < len(label_names) else f"intent_{x}")
    test_df["intent_name"] = test_df["label"].map(lambda x: label_names[int(x)] if 0 <= int(x) < len(label_names) else f"intent_{x}")
else:
    train_df["intent_name"] = train_df["label"].map(lambda x: f"intent_{x}")
    test_df["intent_name"] = test_df["label"].map(lambda x: f"intent_{x}")

print(f"Train completo: {len(train_all):,} | test completo: {len(test_all):,}")
print(f"Muestra usada: train={len(train_df):,} | test={len(test_df):,} | clases={train_df['label'].nunique()}")
display(train_df[["text", "label", "intent_name"]].head(10))
display(train_df["intent_name"].value_counts().head(15).rename_axis("intención").to_frame("n_train"))

## 3. Modelos elegidos

Se comparan tres codificadores multilingües de oraciones. Todos generan vectores semánticos y declaran licencia Apache 2.0 en sus tarjetas de modelo. Las dimensiones sirven como indicador práctico del tamaño de la representación, pero el notebook mide el número real de parámetros, el tiempo de carga, la latencia y la memoria en el entorno donde se ejecute.

| Modelo | Dimensión publicada | Perfil preliminar | Licencia declarada |
|---|---:|---|---|
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | Ligero | Apache-2.0 |
| `distiluse-base-multilingual-cased-v2` | 512 | Intermedio | Apache-2.0 |
| `paraphrase-multilingual-mpnet-base-v2` | 768 | Más pesado | Apache-2.0 |

Los tres reciben exactamente los mismos textos de entrenamiento y prueba. Para cada modelo se entrena el mismo clasificador `LogisticRegression`; las características aprendidas por los codificadores se mantienen congeladas.

In [ ]:
MODEL_SPECS = [
    {"id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "name": "MiniLM-L12-v2", "dim": 384, "license": "Apache-2.0", "profile": "Ligero"},
    {"id": "sentence-transformers/distiluse-base-multilingual-cased-v2", "name": "DistilUSE multilingual v2", "dim": 512, "license": "Apache-2.0", "profile": "Intermedio"},
    {"id": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2", "name": "MPNet multilingual v2", "dim": 768, "license": "Apache-2.0", "profile": "Más pesado"},
]
print(pd.DataFrame(MODEL_SPECS)[["name", "id", "dim", "profile", "license"]].to_string(index=False))

## 4. Funciones de evaluación

Métricas reportadas:

- **Accuracy, precision macro, recall macro y F1 macro:** calidad de clasificación; las métricas macro ponderan por igual las 77 intenciones.
- **Tiempo amortizado por lote:** tiempo total al codificar la partición de prueba en lotes, dividido entre el número de tickets. Es una medida de rendimiento en lote, no latencia de una llamada individual.
- **Latencia individual p50/p95/p99:** mide una llamada por ticket (lote de tamaño 1), tras un calentamiento; incluye tokenización, codificador y clasificador, pero excluye carga inicial del modelo, red y cola de solicitudes. p50 representa la mediana; p95 y p99 aproximan la cola de latencia. Con 300 observaciones el p99 es orientativo, por lo que se debe repetir con más llamadas en un entorno de despliegue.
- **Recursos:** número de parámetros y memoria máxima aproximada utilizada en GPU.

La medición individual es secuencial, se realiza en una muestra reproducible y depende de la GPU, longitud del texto, versiones de librerías y carga compartida de Colab. No equivale automáticamente al tiempo de respuesta extremo a extremo de una aplicación desplegada.

In [ ]:
def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def n_parameters(model):
    return int(sum(p.numel() for p in model.parameters()))

def benchmark_single_ticket_latency(encoder, clf, texts, sample_size=LATENCY_SAMPLE_SIZE,
                                    warmup_samples=LATENCY_WARMUP_SAMPLES):
    """Mide una inferencia aislada por llamada, incluyendo encoder + clasificador."""
    if len(texts) == 0:
        return {"latency_single_sample_n": 0, "latency_single_mean_ms": np.nan,
                "latency_single_p50_ms": np.nan, "latency_single_p95_ms": np.nan,
                "latency_single_p99_ms": np.nan}
    rng = np.random.default_rng(SEED)
    n = min(sample_size, len(texts))
    selected = rng.choice(len(texts), size=n, replace=False)
    sample_texts = [str(texts[int(i)]) for i in selected]

    # Calentamiento para excluir del registro la inicialización/carga inicial de kernels.
    for text in sample_texts[:min(warmup_samples, len(sample_texts))]:
        emb = encoder.encode([text], batch_size=1, show_progress_bar=False,
                             convert_to_numpy=True, normalize_embeddings=True)
        _ = clf.predict_proba(emb)
    sync_device()

    timings_ms = []
    for text in sample_texts:
        sync_device()
        t0 = time.perf_counter()
        emb = encoder.encode([text], batch_size=1, show_progress_bar=False,
                             convert_to_numpy=True, normalize_embeddings=True)
        _ = clf.predict_proba(emb)
        sync_device()
        timings_ms.append((time.perf_counter() - t0) * 1000.0)

    return {
        "latency_single_sample_n": len(timings_ms),
        "latency_single_mean_ms": float(np.mean(timings_ms)),
        "latency_single_p50_ms": float(np.percentile(timings_ms, 50)),
        "latency_single_p95_ms": float(np.percentile(timings_ms, 95)),
        "latency_single_p99_ms": float(np.percentile(timings_ms, 99)),
    }

def run_one_model(spec, train_frame, test_frame):
    model_id = spec["id"]
    print("\n" + "="*88 + "\nModelo:", model_id)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    encoder = SentenceTransformer(model_id, device=device)
    sync_device()
    model_load_s = time.perf_counter() - t0
    params = n_parameters(encoder)

    # Embeddings en lote, para comparar calidad y rendimiento de procesamiento por lote.
    sync_device(); t1 = time.perf_counter()
    X_train = encoder.encode(train_frame["text"].tolist(), batch_size=BATCH_SIZE,
                             show_progress_bar=True, convert_to_numpy=True,
                             normalize_embeddings=True, device=device)
    sync_device(); train_embed_s = time.perf_counter() - t1

    sync_device(); t2 = time.perf_counter()
    X_test = encoder.encode(test_frame["text"].tolist(), batch_size=BATCH_SIZE,
                            show_progress_bar=True, convert_to_numpy=True,
                            normalize_embeddings=True, device=device)
    sync_device(); test_embed_s = time.perf_counter() - t2

    clf = LogisticRegression(max_iter=300, class_weight="balanced", random_state=SEED)
    t3 = time.perf_counter()
    clf.fit(X_train, train_frame["label"].to_numpy())
    fit_s = time.perf_counter() - t3

    t4 = time.perf_counter()
    y_pred = clf.predict(X_test)
    pred_s = time.perf_counter() - t4
    y_true = test_frame["label"].to_numpy()
    single_latency = benchmark_single_ticket_latency(encoder, clf, test_frame["text"].tolist())

    labels = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    result = {
        "model": spec["name"], "model_id": model_id,
        "embedding_dim": spec["dim"], "license": spec["license"],
        "parameters": params, "parameters_millions": params / 1e6,
        "train_samples": len(train_frame), "test_samples": len(test_frame),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "model_load_seconds": model_load_s,
        "train_embedding_seconds": train_embed_s,
        "classifier_fit_seconds": fit_s,
        "test_embedding_seconds": test_embed_s,
        "classifier_predict_seconds": pred_s,
        "batch_amortized_ms_per_ticket": 1000 * (test_embed_s + pred_s) / max(1, len(test_frame)),
        **single_latency,
        "device": device,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "gpu_peak_allocated_mb": torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else np.nan,
    }
    pred_frame = test_frame[["text", "label", "intent_name"]].copy()
    pred_frame = pred_frame.rename(columns={"label":"label_true", "intent_name":"intent_true"})
    pred_frame["label_pred"] = y_pred
    if label_names:
        pred_frame["intent_pred"] = pred_frame["label_pred"].map(lambda x: label_names[int(x)] if 0 <= int(x) < len(label_names) else f"intent_{x}")
    else:
        pred_frame["intent_pred"] = pred_frame["label_pred"].map(lambda x: f"intent_{x}")
    prob = clf.predict_proba(X_test)
    pred_frame["confidence"] = prob.max(axis=1)
    pred_frame["model"] = spec["name"]
    slug = spec["name"].lower().replace(" ", "_").replace("-", "_")
    pred_frame.to_csv(OUT_DIR / f"predictions_{slug}.csv", index=False, encoding="utf-8-sig")
    report = classification_report(y_true, y_pred, labels=labels, zero_division=0,
                                   output_dict=True, target_names=[str(x) for x in labels])
    pd.DataFrame(report).transpose().to_csv(OUT_DIR / f"classification_report_{slug}.csv", encoding="utf-8-sig")
    del encoder, clf, X_train, X_test, prob
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 5. Ejecución de los tres modelos

La celda siguiente descarga los pesos la primera vez y puede tomar varios minutos. Usa siempre la misma muestra balanceada y las mismas métricas.

In [ ]:
results = []
for spec in MODEL_SPECS:
    results.append(run_one_model(spec, train_df, test_df))

results_df = pd.DataFrame(results).sort_values(
    ["f1_macro", "latency_single_p95_ms"], ascending=[False, True]
).reset_index(drop=True)
results_path = OUT_DIR / "model_comparison.csv"
results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
display(results_df[["model", "parameters_millions", "accuracy", "precision_macro", "recall_macro",
                    "f1_macro", "batch_amortized_ms_per_ticket", "latency_single_p50_ms",
                    "latency_single_p95_ms", "latency_single_p99_ms", "gpu_peak_allocated_mb"]].round(4))
print("Resultados guardados en:", results_path.resolve())

## 6. Visualización comparativa

In [ ]:
# Gráfica 1: métricas de clasificación
plot_metrics = results_df.set_index("model")[["accuracy", "precision_macro", "recall_macro", "f1_macro"]]
ax = plot_metrics.plot(kind="bar", figsize=(12, 6), ylim=(0, 1), rot=15)
ax.set_title("Comparación de métricas por modelo")
ax.set_ylabel("Puntuación (0–1)")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "metrics_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

# Gráfica 2: latencia individual (p50 y p95); esta es la comparación principal para solicitudes de una en una.
lat = results_df.set_index("model")[["latency_single_p50_ms", "latency_single_p95_ms"]].sort_values("latency_single_p95_ms")
ax = lat.plot(kind="barh", figsize=(10, 5))
ax.set_title("Latencia de inferencia individual por ticket")
ax.set_xlabel("Milisegundos por ticket (lote de tamaño 1; sin tiempo de carga del modelo)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "latency_single_ticket_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

# Gráfica 3: tiempo por ticket amortizado sobre procesamiento en lotes, útil para rendimiento batch.
lat_batch = results_df.set_index("model")["batch_amortized_ms_per_ticket"].sort_values()
ax = lat_batch.plot(kind="barh", figsize=(10, 4))
ax.set_title("Tiempo amortizado por ticket en procesamiento por lotes")
ax.set_xlabel("Milisegundos por ticket (codificador + clasificador; ejecución en lote)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "latency_batch_amortized_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Selección condicionada por evidencia

Regla de decisión inicial para el prototipo: **F1 macro ≥ 0.75** y **latencia individual p95 ≤ 250 ms**. El umbral de latencia es una propuesta que debe confirmar el equipo de Operación. La latencia se mide con el modelo ya cargado, una petición por llamada, después de un calentamiento; se reporta sobre una muestra reproducible. Si ningún modelo satisface ambos criterios, no se considera aprobado para piloto: se debe investigar el entorno, los datos o la arquitectura.

In [ ]:
qualified = results_df[(results_df["f1_macro"] >= MIN_F1_MACRO) &
                       (results_df["latency_single_p95_ms"] <= MAX_SINGLE_TICKET_P95_MS)]
if len(qualified):
    chosen = qualified.sort_values(["f1_macro", "latency_single_p95_ms"], ascending=[False, True]).iloc[0]
    print("CANDIDATO EXPLORATORIO PARA PILOTO:", chosen["model"])
    print(f"F1 macro={chosen['f1_macro']:.3f}; latencia individual p50={chosen['latency_single_p50_ms']:.1f} ms, "
          f"p95={chosen['latency_single_p95_ms']:.1f} ms, p99={chosen['latency_single_p99_ms']:.1f} ms")
    print("Antes de despliegue: revisar intenciones críticas, umbral de confianza, tickets mexicanos reales y capacidad operativa.")
else:
    chosen = results_df.sort_values(["f1_macro", "latency_single_p95_ms"], ascending=[False, True]).iloc[0]
    print("NINGÚN MODELO CUMPLE SIMULTÁNEAMENTE LOS UMBRALES INICIALES.")
    print("Mejor candidato exploratorio por F1 macro:", chosen["model"])
    print(f"F1 macro={chosen['f1_macro']:.3f}; latencia individual p95={chosen['latency_single_p95_ms']:.1f} ms")
    print("No se recomienda desplegar aún; revisar medición, arquitectura y umbrales con Operación.")

## 8. Inferencia de ejemplo y escalamiento humano

La confianza es la probabilidad máxima estimada por la regresión logística; **no es una garantía de corrección ni debe tratarse como probabilidad calibrada**. Se recomienda que el asistente solo sugiera una cola cuando supere un umbral validado y que, por política, las categorías sensibles siempre pasen por revisión humana. La celda siguiente deja explícita esa regla de ejemplo; el listado debe ser aprobado por el negocio, Riesgos y Cumplimiento.

In [ ]:
# Se recarga el codificador candidato y se vuelve a ajustar el clasificador con el mismo conjunto de entrenamiento.
chosen_spec = next(s for s in MODEL_SPECS if s["name"] == chosen["model"])
encoder = SentenceTransformer(chosen_spec["id"], device=device)
Xtr = encoder.encode(train_df["text"].tolist(), batch_size=BATCH_SIZE, show_progress_bar=True,
                     convert_to_numpy=True, normalize_embeddings=True)
Xte = encoder.encode(test_df["text"].tolist(), batch_size=BATCH_SIZE, show_progress_bar=True,
                     convert_to_numpy=True, normalize_embeddings=True)
clf = LogisticRegression(max_iter=300, class_weight="balanced", random_state=SEED)
clf.fit(Xtr, train_df["label"].to_numpy())

# Política inicial de escalamiento: editar y validar con Operación/Riesgos antes de cualquier piloto.
SENSITIVE_INTENTS = {
    "card_payment_not_recognised", "cash_withdrawal_not_recognised",
    "direct_debit_payment_not_recognised", "compromised_card",
    "lost_or_stolen_card", "lost_or_stolen_phone", "transaction_charged_twice",
    "transfer_not_received_by_recipient"
}
DEFAULT_CONFIDENCE_THRESHOLD = 0.65  # valor exploratorio; elegir con la tabla de cobertura/errores

def classify_ticket(text, confidence_threshold=DEFAULT_CONFIDENCE_THRESHOLD):
    emb = encoder.encode([text], batch_size=1, show_progress_bar=False,
                         convert_to_numpy=True, normalize_embeddings=True)
    prob = clf.predict_proba(emb)[0]
    idx = int(np.argmax(prob))
    label_id = int(clf.classes_[idx])
    intent = label_names[label_id] if label_names and label_id < len(label_names) else f"intent_{label_id}"
    confidence = float(prob[idx])
    if intent in SENSITIVE_INTENTS:
        route = "Revisión humana obligatoria: intención sensible"
    elif confidence < confidence_threshold:
        route = "Revisión humana: confianza por debajo del umbral"
    else:
        route = "Cola sugerida; confirmar con agente"
    return {"ticket": text, "intent_id": label_id, "intent": intent,
            "confidence": confidence, "confidence_threshold": confidence_threshold,
            "route": route}

example = "No reconozco un cargo en mi tarjeta y necesito ayuda para reportarlo."
print(json.dumps(classify_ticket(example), ensure_ascii=False, indent=2))

## 9. Evaluación del umbral de confianza y cobertura de revisión humana

Esta sección evalúa varios umbrales con las predicciones de prueba. **Cobertura** es la proporción de tickets que superarían el umbral; los restantes quedarían en revisión humana. La exactitud selectiva mide la proporción correcta dentro de los tickets aceptados por el umbral (no es la precision macro por clase). También se calcula una cobertura segura que excluye las intenciones incluidas en `SENSITIVE_INTENTS`, que deben revisarse siempre.

La tabla permite comparar el costo operativo (cuántos tickets requieren revisión) con el riesgo observado de sugerir una intención incorrecta. Los resultados son retrospectivos y usan un benchmark traducido; no sirven como autorización para automatizar tickets reales. El umbral debe validarse en un conjunto separado de tickets mexicanos, y calibrarse si es necesario.

In [ ]:
THRESHOLDS = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

threshold_rows = []
calibration_rows = []
prediction_frames = {}
for spec in MODEL_SPECS:
    slug = spec["name"].lower().replace(" ", "_").replace("-", "_")
    pred_path = OUT_DIR / f"predictions_{slug}.csv"
    if not pred_path.exists():
        print(f"No se encuentra {pred_path}; asegúrate de haber ejecutado la comparación antes de esta sección.")
        continue
    pred_df = pd.read_csv(pred_path)
    prediction_frames[spec["name"]] = pred_df
    y_true = pred_df["label_true"].to_numpy()
    y_pred = pred_df["label_pred"].to_numpy()
    conf = pred_df["confidence"].astype(float).to_numpy()
    correct = (y_true == y_pred)
    pred_intents = pred_df["intent_pred"].astype(str).to_numpy()
    is_sensitive = np.isin(pred_intents, list(SENSITIVE_INTENTS))

    # ECE de etiqueta principal: diferencia ponderada entre confianza promedio y exactitud observada por intervalo.
    ece = 0.0
    n_bins = 10
    bin_rows = []
    for b in range(n_bins):
        low, high = b / n_bins, (b + 1) / n_bins
        mask = (conf >= low) & ((conf < high) if b < n_bins - 1 else (conf <= high))
        if mask.any():
            bin_accuracy = float(correct[mask].mean())
            bin_confidence = float(conf[mask].mean())
            ece += float(mask.mean()) * abs(bin_accuracy - bin_confidence)
            bin_rows.append({"model": spec["name"], "bin_low": low, "bin_high": high,
                             "n": int(mask.sum()), "mean_confidence": bin_confidence,
                             "observed_accuracy": bin_accuracy})
    calibration_rows.append({"model": spec["name"], "n_test": len(pred_df),
                             "accuracy": float(correct.mean()),
                             "mean_top_confidence": float(conf.mean()),
                             "median_top_confidence": float(np.median(conf)),
                             "ece_top_label_10_bins": float(ece)})

    for threshold in THRESHOLDS:
        accepted = conf >= threshold
        safe_accepted = accepted & (~is_sensitive)
        for mode, mask in [("confidence_only", accepted), ("sensitive_intents_to_human", safe_accepted)]:
            accepted_n = int(mask.sum())
            selective_accuracy = float(correct[mask].mean()) if accepted_n else np.nan
            threshold_rows.append({
                "model": spec["name"], "threshold": threshold, "policy": mode,
                "n_test": int(len(pred_df)), "accepted_n": accepted_n,
                "coverage": float(mask.mean()), "human_review_rate": float(1 - mask.mean()),
                "accepted_accuracy": selective_accuracy,
                "accepted_error_rate": float(1 - selective_accuracy) if accepted_n else np.nan,
                "incorrect_suggestions_as_pct_all_tickets": float(((~correct) & mask).mean()),
                "sensitive_predicted_n": int(is_sensitive.sum())
            })

threshold_df = pd.DataFrame(threshold_rows)
threshold_path = OUT_DIR / "confidence_threshold_analysis.csv"
threshold_df.to_csv(threshold_path, index=False, encoding="utf-8-sig")
calibration_df = pd.DataFrame(calibration_rows)
calibration_path = OUT_DIR / "top_label_calibration_summary.csv"
calibration_df.to_csv(calibration_path, index=False, encoding="utf-8-sig")
if bin_rows:
    # Se guarda la tabla de bins por modelo para inspección posterior.
    all_bin_rows = []
    for model_name, pdf in prediction_frames.items():
        c = pdf["confidence"].astype(float).to_numpy()
        ok = (pdf["label_true"].to_numpy() == pdf["label_pred"].to_numpy())
        for b in range(10):
            low, high = b / 10, (b + 1) / 10
            mask = (c >= low) & ((c < high) if b < 9 else (c <= high))
            if mask.any():
                all_bin_rows.append({"model": model_name, "bin_low": low, "bin_high": high,
                                     "n": int(mask.sum()), "mean_confidence": float(c[mask].mean()),
                                     "observed_accuracy": float(ok[mask].mean())})
    pd.DataFrame(all_bin_rows).to_csv(OUT_DIR / "top_label_calibration_bins.csv", index=False, encoding="utf-8-sig")

display(threshold_df[(threshold_df["policy"] == "sensitive_intents_to_human") &
                     (threshold_df["threshold"].isin([0.50, 0.60, 0.65, 0.70, 0.75]))]
        [["model", "threshold", "coverage", "human_review_rate", "accepted_n",
          "accepted_accuracy", "accepted_error_rate"]]
        .sort_values(["threshold", "model"]).round(4))
display(calibration_df.round(4))

# Curva de confiabilidad de la confianza de etiqueta principal para el modelo candidato.
chosen_predictions = prediction_frames.get(chosen_spec["name"])
if chosen_predictions is not None:
    c = chosen_predictions["confidence"].astype(float).to_numpy()
    ok = (chosen_predictions["label_true"].to_numpy() == chosen_predictions["label_pred"].to_numpy())
    curve_x, curve_y = [], []
    for b in range(10):
        low, high = b / 10, (b + 1) / 10
        mask = (c >= low) & ((c < high) if b < 9 else (c <= high))
        if mask.sum() >= 10:
            curve_x.append(float(c[mask].mean()))
            curve_y.append(float(ok[mask].mean()))
    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.plot([0, 1], [0, 1], linestyle="--", label="Confianza ideal")
    ax.plot(curve_x, curve_y, marker="o", label=chosen_spec["name"])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Confianza media del modelo")
    ax.set_ylabel("Exactitud observada")
    ax.set_title("Calibración de la predicción principal (10 intervalos)")
    ax.grid(alpha=0.25); ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "confidence_reliability_curve.png", dpi=160, bbox_inches="tight")
    plt.show()

print("Umbrales guardados en:", threshold_path.resolve())
print("Resumen de calibración guardado en:", calibration_path.resolve())
print("Nota: un ECE alto indica que el score de confianza no coincide bien con la tasa de acierto por intervalo; no interpretes la confianza como probabilidad calibrada.")

## 10. Conclusiones y limitaciones

Al concluir la ejecución, registra el equilibrio entre F1 macro, recall de intenciones críticas, latencia individual p95/p99, cobertura de sugerencias y porcentaje de revisión humana. No selecciones el modelo únicamente por accuracy global.

**Controles mínimos antes de cualquier piloto con clientes o empleados:**

- **Validación de dominio:** el conjunto está traducido y no representa necesariamente el español mexicano ni los procesos de un banco concreto.
- **Privacidad:** nunca subir al Hub ni al repositorio tickets reales, tokens, nombres, cuentas, teléfonos o información sensible.
- **Confianza:** el umbral de la tabla es exploratorio. Validarlo en una partición separada; evaluar calibración, cobertura y errores por intención.
- **Intenciones sensibles:** mantener revisión humana obligatoria para robo, tarjetas comprometidas y cargos no reconocidos, según una política aprobada por Riesgos y Operación.
- **Evaluación por clase:** revisar recall/precision y confusiones; el soporte por clase es limitado y unas pocas observaciones cambian las métricas.
- **Mantenimiento:** versionar dataset, dependencias y resultados; vigilar deriva de vocabulario y volver a evaluar periódicamente.
- **Latencia:** el notebook mide p50/p95/p99 con una llamada por ticket después del calentamiento. Los CSV ya generados corresponden a una medición amortizada por lotes; para obtener los nuevos percentiles hay que ejecutar las celdas de comparación actualizadas en Colab. Repetir luego en el entorno de despliegue con cargas reales y medición extremo a extremo.
- **Licencias:** las tarjetas de los tres modelos declaran Apache-2.0. Revisar de forma independiente las condiciones de la versión traducida del dataset antes de un uso comercial.

### Archivos generados

La carpeta `results/` incluirá la comparación global, latencia individual, predicciones y reportes por modelo, la evaluación de umbrales (`confidence_threshold_analysis.csv`), resumen de calibración (`top_label_calibration_summary.csv`), intervalos de calibración y gráficas. Revisa los archivos antes de compartirlos y evita incluir datos sensibles.